# Setup Mario (Python 3.7.1)

In [ ]:
# Installing game environment
%pip install gym_super_mario_bros==7.3.0 nes_py
%pip install gym==0.23.1

In [ ]:
#! Import the game
import gym_super_mario_bros
#! Import the Joypad wrapper
from nes_py.wrappers import JoypadSpace
#! Import the SIMPLIFIED controls
from gym_super_mario_bros.actions import SIMPLE_MOVEMENT
#! Import time for visualization
import time

In [ ]:
# Reducing the possible moves for our AI to boost the learning process
SIMPLE_MOVEMENT

In [ ]:
#! Setup game
env = gym_super_mario_bros.make('SuperMarioBros-v0')
env = JoypadSpace(env, SIMPLE_MOVEMENT)

In [ ]:
# What our game will look like
env.observation_space.shape

In [ ]:
# env without SIMPLE_MOVEMENT would have 256 discrete moves
env.action_space

In [ ]:
# Getting random actions
SIMPLE_MOVEMENT[env.action_space.sample()]

In [ ]:
# Create a flag - restart or not
done = True
# Loop through each frame in the game
for step in range(100000):
    # Start the game to begin with
    if done:
        # Start the game
        env.reset()
    # Do random actions
    state, reward, done, info = env.step(env.action_space.sample())
    # Show the game on the screen
    env.render()

    # Add a delay of 0.1 seconds
    time.sleep(0.0001)

# Close the game
env.close()

In [ ]:
env.close()

In [ ]:
state = env.reset()

In [ ]:
env.step(1)[3]

# Preprocess Environment

In [ ]:
# Install pytorch
%pip install torch==1.13.0+cu117 torchvision==0.14.0+cu117 torchaudio==0.13.0 --extra-index-url https://download.pytorch.org/whl/cu117

In [ ]:
# Install stable baselines for RL stuff
%pip install stable-baselines3[extra]==1.3.0

In [ ]:
#! Import Frame Stacker Wrapper and GrayScaling Wrapper
from gym.wrappers import GrayScaleObservation
#! Import Vectorization Wrappers
from stable_baselines3.common.vec_env import VecFrameStack, DummyVecEnv
#! Import Matplotlib to show the impact of frame stacking
from matplotlib import pyplot as plt

In [ ]:
#! 1. Create the base environment
env = gym_super_mario_bros.make('SuperMarioBros-v0')
#! 2. Simplify the controls
env = JoypadSpace(env, SIMPLE_MOVEMENT)
#! 3. Grayscale
env = GrayScaleObservation(env, keep_dim=True)
#! 4. Wrap inside the Dummy Environment
env = DummyVecEnv([lambda: env])
#! 5. Stack the frames
env = VecFrameStack(env, 4, channels_order='last')

In [ ]:
#! Reset game state
state = env.reset()

In [ ]:
#! Define game variables
state, reward, done, info = env.step([env.action_space.sample()])

In [ ]:
# Use MatplotLib to show the game frame
plt.imshow(state[0])

In [ ]:
plt.figure(figsize=(20,16))
for idx in range(state.shape[3]):
    plt.subplot(1,4,idx+1)
    plt.imshow(state[0][:,:,idx])
plt.show()

# Train the RL Model

In [ ]:
#! Import os for file path management
import os
#! Import PPO for algorithms
from stable_baselines3 import PPO
#! Import Base Callback for saving models
from stable_baselines3.common.callbacks import BaseCallback

In [ ]:
#! Creating class for logging
class TrainAndLoggingCallback(BaseCallback):

    def __init__(self, check_freq, save_path, verbose=1):
        super(TrainAndLoggingCallback, self).__init__(verbose)
        self.check_freq = check_freq
        self.save_path = save_path

    def _init_callback(self):
        if self.save_path is not None:
            os.makedirs(self.save_path, exist_ok=True)

    def _on_step(self):
        if self.n_calls % self.check_freq == 0:
            model_path = os.path.join(self.save_path, 'best_model_{}'.format(self.n_calls))
            self.model.save(model_path)

        return True

In [ ]:
#! Define logging path
CHECKPOINT_DIR = './train/'
LOG_DIR = './logs/'

In [ ]:
#! Setup model saving callback (adjust check_freq)
callback = TrainAndLoggingCallback(check_freq=5000000, save_path=CHECKPOINT_DIR)

In [ ]:
#! This is the AI model (adjust learning_rate, n_steps)
model = PPO('CnnPolicy', env, verbose=1, tensorboard_log=LOG_DIR, learning_rate=0.00001, n_steps=512)

In [ ]:
# Train the AI model, this is where the AI model starts to learn (adjust total_timesteps)
model.learn(total_timesteps=30000000, callback=callback)

# Test it Out

In [ ]:
# Installing Tensorboad for visualization
%pip install tensorboard
# py -m tensorboard.main --logdir="C:\Users\bened\OneDrive - DHBW Lörrach\Desktop\Coding\Grundlagen DS & KI\Project_Mario\logs"

In [ ]:
# Load model (adjust path)
model = PPO.load('C:/Users/bened/OneDrive - DHBW Lörrach/1. Semester/Grundlagen DS & KI/Project-Mario/train/best_model_15000000.zip')

In [ ]:
# Reset state
state = env.reset()

In [ ]:
# Start the game
state = env.reset()
# Loop through the game
while True:
    action, _ = model.predict(state)
    state, reward, done, info = env.step(action)
    env.render()

    # Add a delay of 0.1 seconds
    time.sleep(0.0001)

In [ ]:
env.close()